# Neo4j Knowledge Graph Browser

**Read-only notebook for exploring the knowledge graph**

Features:
1. Search and browse knowledge units
2. Visualize graph structure
3. Run Cypher queries for analysis
4. Export reports and statistics

**Note:** To edit content, modify YAML files in `/home/mike/0bsidian/skuel/domains/` and re-sync.

In [ ]:
# Setup and imports
import asyncio
import json
import os
from pathlib import Path
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
from neo4j import AsyncGraphDatabase
import pandas as pd

# Configuration — reads from environment, with development defaults
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")

if not NEO4J_PASSWORD:
    try:
        from core.config.credential_store import get_credential
        NEO4J_PASSWORD = get_credential("NEO4J_PASSWORD", fallback_to_env=True)
    except Exception:
        raise RuntimeError("NEO4J_PASSWORD not set. Export it or use the credential store.")

# Initialize driver
driver = AsyncGraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

print(f"✅ Connected to Neo4j at {NEO4J_URI}")
print("📖 Read-only mode - edit YAML files to make changes")

## 1. Graph Statistics

In [ ]:
async def get_graph_stats():
    """Get overview statistics of the knowledge graph."""
    stats_query = """
    MATCH (ku:KnowledgeUnit)
    OPTIONAL MATCH (ku)-[r]->()
    RETURN 
        count(DISTINCT ku) as total_units,
        count(DISTINCT ku.domain) as domains,
        count(r) as total_relationships,
        collect(DISTINCT ku.complexity) as complexities
    """
    
    async with driver.session() as session:
        result = await session.run(stats_query)
        record = await result.single()
        
        print("📊 Knowledge Graph Statistics")
        print("="*50)
        print(f"Total Knowledge Units: {record['total_units']}")
        print(f"Domains: {record['domains']}")
        print(f"Total Relationships: {record['total_relationships']}")
        print(f"Complexity Levels: {', '.join([c for c in record['complexities'] if c])}")
        print("="*50)

await get_graph_stats()

## 2. Search and Browse Knowledge Units

In [ ]:
async def search_knowledge_units(query: str = "", domain: str = None, complexity: str = None):
    """Search for knowledge units with filters."""
    where_clauses = ["ku.title CONTAINS $query OR ku.uid CONTAINS $query"]
    params = {"query": query}
    
    if domain:
        where_clauses.append("ku.domain = $domain")
        params["domain"] = domain
    
    if complexity:
        where_clauses.append("ku.complexity = $complexity")
        params["complexity"] = complexity
    
    search_query = f"""
    MATCH (ku:KnowledgeUnit)
    WHERE {' AND '.join(where_clauses)}
    RETURN ku.uid as uid, 
           ku.title as title, 
           ku.domain as domain, 
           ku.complexity as complexity,
           ku.quality_score as quality,
           size(ku.tags) as tag_count
    ORDER BY ku.title
    LIMIT 50
    """
    
    async with driver.session() as session:
        result = await session.run(search_query, params)
        records = await result.data()
        return records

# Search widget
search_input = widgets.Text(
    placeholder='Search by title or UID...',
    description='Search:',
    style={'description_width': 'initial'}
)

domain_filter = widgets.Dropdown(
    options=[('All', None), ('personal', 'personal'), ('technology', 'technology')],
    description='Domain:',
)

complexity_filter = widgets.Dropdown(
    options=[('All', None), ('basic', 'basic'), ('intermediate', 'intermediate'), ('advanced', 'advanced')],
    description='Complexity:',
)

search_output = widgets.Output()

def on_search(change):
    with search_output:
        search_output.clear_output()
        results = asyncio.run(search_knowledge_units(
            search_input.value,
            domain_filter.value,
            complexity_filter.value
        ))
        
        if results:
            df = pd.DataFrame(results)
            display(df)
        else:
            print("No results found")

search_button = widgets.Button(description='Search', button_style='primary')
search_button.on_click(on_search)

display(widgets.VBox([
    search_input,
    widgets.HBox([domain_filter, complexity_filter]),
    search_button,
    search_output
]))

## 3. View Knowledge Unit Details

In [ ]:
async def view_knowledge_unit(uid: str):
    """View detailed information about a knowledge unit."""
    detail_query = """
    MATCH (ku:KnowledgeUnit {uid: $uid})
    OPTIONAL MATCH (ku)-[:PREREQUISITE]->(prereq)
    OPTIONAL MATCH (ku)-[:ENABLES]->(enabled)
    OPTIONAL MATCH (ku)-[:RELATED_TO]->(related)
    RETURN ku,
           collect(DISTINCT {uid: prereq.uid, title: prereq.title}) as prerequisites,
           collect(DISTINCT {uid: enabled.uid, title: enabled.title}) as enables,
           collect(DISTINCT {uid: related.uid, title: related.title}) as related_to
    """
    
    async with driver.session() as session:
        result = await session.run(detail_query, {"uid": uid})
        record = await result.single()
        
        if not record:
            print(f"❌ Knowledge unit not found: {uid}")
            return
        
        ku = dict(record['ku'])
        
        print(f"\n📚 {ku.get('title')}")
        print("="*80)
        print(f"UID: {ku.get('uid')}")
        print(f"Domain: {ku.get('domain')}")
        print(f"Complexity: {ku.get('complexity')}")
        print(f"Quality Score: {ku.get('quality_score')}")
        print(f"Tags: {', '.join(ku.get('tags', []))}")
        
        print("\n🔗 Relationships:")
        print("-"*80)
        
        prereqs = [p for p in record['prerequisites'] if p['uid']]
        if prereqs:
            print(f"\n📖 Prerequisites ({len(prereqs)}):")
            for p in prereqs:
                print(f"  - {p['title']} ({p['uid']})")
        
        enables = [e for e in record['enables'] if e['uid']]
        if enables:
            print(f"\n🚀 Enables ({len(enables)}):")
            for e in enables:
                print(f"  - {e['title']} ({e['uid']})")
        
        related = [r for r in record['related_to'] if r['uid']]
        if related:
            print(f"\n🔄 Related To ({len(related)}):")
            for r in related:
                print(f"  - {r['title']} ({r['uid']})")
        
        print("\n📄 Content Preview:")
        print("-"*80)
        content = ku.get('content', '')
        preview = content[:500] + '...' if len(content) > 500 else content
        print(preview)
        print("="*80)

# Detail viewer widget
uid_input = widgets.Text(
    placeholder='Enter UID (e.g., ku:note-taking-basics)',
    description='UID:',
    style={'description_width': 'initial'},
    layout={'width': '500px'}
)

detail_output = widgets.Output()

def on_view_detail(change):
    with detail_output:
        detail_output.clear_output()
        if uid_input.value:
            asyncio.run(view_knowledge_unit(uid_input.value))

view_button = widgets.Button(description='View Details', button_style='info')
view_button.on_click(on_view_detail)

display(widgets.VBox([uid_input, view_button, detail_output]))

## 4. Visualize Knowledge Paths

In [ ]:
async def get_learning_path(start_uid: str, depth: int = 3):
    """Get the learning path from a starting knowledge unit."""
    path_query = """
    MATCH path = (start:KnowledgeUnit {uid: $start_uid})-[:PREREQUISITE*0..{depth}]->(prereq)
    RETURN path
    LIMIT 50
    """.replace("{depth}", str(depth))
    
    async with driver.session() as session:
        result = await session.run(path_query, {"start_uid": start_uid})
        records = await result.data()
        
        print(f"\n🗺️ Learning Path from {start_uid}")
        print("="*80)
        print(f"Found {len(records)} prerequisite paths (depth: {depth})")
        print("\nTo visualize, run this query in Neo4j Browser:")
        print("-"*80)
        print(f"MATCH path = (start:KnowledgeUnit {{uid: '{start_uid}'}})-[:PREREQUISITE*0..{depth}]->(prereq)")
        print("RETURN path")
        print("="*80)

# Path visualization widget
path_uid_input = widgets.Text(
    placeholder='Enter starting UID',
    description='Start UID:',
    style={'description_width': 'initial'},
    layout={'width': '400px'}
)

path_depth = widgets.IntSlider(
    value=3,
    min=1,
    max=5,
    description='Depth:',
)

path_output = widgets.Output()

def on_view_path(change):
    with path_output:
        path_output.clear_output()
        if path_uid_input.value:
            asyncio.run(get_learning_path(path_uid_input.value, path_depth.value))

path_button = widgets.Button(description='View Path', button_style='success')
path_button.on_click(on_view_path)

display(widgets.VBox([
    path_uid_input,
    path_depth,
    path_button,
    path_output
]))

## 5. Custom Cypher Queries

In [ ]:
# Direct Cypher query execution
cypher_editor = widgets.Textarea(
    value="""MATCH (ku:KnowledgeUnit)
RETURN ku.domain as domain, 
       count(ku) as count,
       avg(ku.quality_score) as avg_quality
ORDER BY count DESC""",
    description='Cypher Query:',
    layout={'width': '100%', 'height': '150px'},
    style={'description_width': 'initial'}
)

query_output = widgets.Output()

async def execute_cypher():
    """Execute custom Cypher query."""
    query = cypher_editor.value
    
    try:
        async with driver.session() as session:
            result = await session.run(query)
            records = await result.data()
            
            with query_output:
                query_output.clear_output()
                if records:
                    df = pd.DataFrame(records)
                    display(df)
                    print(f"\n✅ Returned {len(records)} rows")
                else:
                    print("Query returned no results")
    except Exception as e:
        with query_output:
            query_output.clear_output()
            print(f"❌ Query error: {e}")

execute_button = widgets.Button(description='Execute Query', button_style='primary')
execute_button.on_click(lambda b: asyncio.run(execute_cypher()))

display(cypher_editor)
display(execute_button)
display(query_output)

## 6. Export Reports

In [ ]:
async def export_domain_report(domain: str = None):
    """Export a report of all knowledge units in a domain."""
    where_clause = "WHERE ku.domain = $domain" if domain else ""
    params = {"domain": domain} if domain else {}
    
    report_query = f"""
    MATCH (ku:KnowledgeUnit)
    {where_clause}
    OPTIONAL MATCH (ku)-[:PREREQUISITE]->(prereq)
    OPTIONAL MATCH (ku)-[:ENABLES]->(enabled)
    RETURN ku.uid as uid,
           ku.title as title,
           ku.domain as domain,
           ku.complexity as complexity,
           ku.quality_score as quality,
           ku.tags as tags,
           count(DISTINCT prereq) as prerequisite_count,
           count(DISTINCT enabled) as enables_count
    ORDER BY ku.domain, ku.title
    """
    
    async with driver.session() as session:
        result = await session.run(report_query, params)
        records = await result.data()
        
        if records:
            df = pd.DataFrame(records)
            
            # Save to CSV
            output_file = f"/tmp/knowledge_report_{domain or 'all'}.csv"
            df.to_csv(output_file, index=False)
            
            print(f"\n📊 Knowledge Report")
            print("="*80)
            print(f"Domain: {domain or 'All'}")
            print(f"Total Units: {len(records)}")
            print(f"Exported to: {output_file}")
            print("="*80)
            
            display(df)
            return output_file
        else:
            print("No data to export")
            return None

# Export widget
export_domain = widgets.Dropdown(
    options=[('All Domains', None), ('personal', 'personal'), ('technology', 'technology')],
    description='Domain:',
)

export_output = widgets.Output()

def on_export(change):
    with export_output:
        export_output.clear_output()
        asyncio.run(export_domain_report(export_domain.value))

export_button = widgets.Button(description='Export Report', button_style='warning')
export_button.on_click(on_export)

display(widgets.VBox([export_domain, export_button, export_output]))

## 7. Cleanup

In [ ]:
# Close driver when done
await driver.close()
print("✅ Driver closed")